# Patchout Solution

A spectrogram of audio has:
- x axis: time
- y axis: frequencies
- Pixel intensity: energy at a particular time and frequency

Suppose our spectrogram produces 1,200 tokens. Self-attention needs to calculate relationships between every pair of tokens.
$$
A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)
$$

The attention matrix has shape:
$$
A\in\mathbb{R}^{1200\times1200}
$$

Now suppose we randomly discard 50% of the tokens before the transformer.

$$
N'=600
$$

The new attention matrix has 75% fewer attention scores, since:

$$
\frac{(N/2)^2}{N^2}=\frac14
$$

But here's the interesting research question:

> How can a model learn to recognise an instrument when we deliberately remove parts of its spectrogram?

[Efficient Training of Audio Transformers with Patchout](https://arxiv.org/pdf/2110.05069)

## 1. The Complexity problem of Transformer

Transformer is better than CNN in image classification, but it has drawbacks:
- requires large amounts of training data (Vision Transformer - ViT)
- depends on extensive data augmentation and knowledge distillation (Data-efficient Image Transformers - DeiT)

It is also used to audio spectrogram, with pre-trained models from computer vision and overlapping patches from audio spectrogram.

Original transformer complexity is $O(n^2)$ (boath memory and computation), and the paper introduces:
- Patchout approach to reduce the quadratic complexity
- Disentangle the transformer’s positional encoding into time and frequency positional encoding -> straightforward inference on audio snippets of variable length without the need for fine-tuning or interpolating positional encodings.
- Additional methods for reducing training complexity.


## 2. PaSST - How does spectrogram becomes tokens?

- **ViT**: adds trainable positional encodings as biases, classification embedding.
- **Deit** (Data-efficient Image Transformers): adds distillation embedding.
- **Overlapping** on patches improve performance when training ViT on spectrograms, but increases memory and computation.

Proposed Architecture

![Patchout Architecture](../assets/img/patchout_architecture.png)

1. Patch extraction and linear projection
2. Frequency and time positional encodings (prev works has only one positional encoding for both)
3. Patchout and classification token
4. Flatten sequence and pass through d layers blocks of self-attention.

![Spectrogram example](../assets/img/stft_spectrogram_eg.png)

### 2.1 Complexity Analysis - Why is attention quadratic?

- $O(n^2)$: distance between pair of positions in the input sequence (attention matrix).
- b batches, (n x e) samples, with n:= sequence length, e:= embedding size.
- h attention heads.
- Q, K, V have shape $(n \times \frac{e}{h})$ for each head.
- Attention matrix: $A = QK^T$, then applied softmax: $A = \text{softmax}(\frac{QK^T}{\sqrt{d}})$, shape $(n\times n)$.
- Then multiply $A \times V$ to return value with same shape as input.

### 2.2 Patchout - What exactly gets removed?

- Drop parts of input -> classification on incomplete sequence.
- Input sequence: extract small overlapping patches from the input spectrograms and linearly project them to vectors.
- Augment the patches with both frequency and time encoding.
- When training, we randomly drop parts of the sequence, reducing the sequence length, and effectively regularizing the training process.
- Similar to DropOut, during inference, the whole input sequence is presented to the transformer.
- Different types of Patchout:
  - **Unstructured Patchout** (PaSST-U): select the patches randomly regardless of their position.
  - **Structured Patchout** (PaSST-S): randomly pick some frequency bins/time frames and remove a whole column/row of extracted patches.

### 2.3 Other techniques used

- **Reduce patches' overlap**: reduces # patches, but patchout can work when there's no overlap (PaSST-N).
- **Reduce transformer depth**: reduces # blocks (d). Since we are starting the training from models pre-trained on Imagenet, we remove every other self-attention block (PaSST-L with `d=7` self-attention blocks and 50M params - compared to 87M full model).

## 3. How do experiments validate the claims?

### 3.1 Preprocessing and Training Setup

- Mono audio with a sampling rate of 32kHz > Extract Mel features from a window of 25ms with a hop length of 10ms, resulting in 128 mel bands:
  - `n_fft` = 32000 * 0.025 = 800 samples
  - `hop_length` = 32000 * 0.01 = 320 samples
  - STFT bins = n_fft / 2 + 1 = 401 bins
  - `d` = dimensions in Mel feature matrix = 128.
- **Importance sampling**:  assign a sampling weight to each sample proportional to the inverse frequency of its label $\frac{1}{\text{freq}(\text{label}) + 100}$.
- Train set = 1,893,693 segments
- Evaluate set = 18,951 audio clips
- For each epoch, we sample 200k samples from the full 2M Audioset without replacement.
- AdamW optimiser, weight decay = $10^-4$, max lr = $10^-5$.
- Learning rate decay from epoch 50 to 100, dropping the learning rate to $10^{-7}$ and fine-tune the model for a further 20 epochs.

### 3.2 Pre-trained Model

- DeiT B↑384: the model was fine-tuned at a higher image resolution than its original training resolution. (B:= Base model size architecture, (384×384 pixels))
- stripped-down ViT-hug224: remove half of the self-attention blocks, its depth was reduced to only 16 blocks.

### 3.3 Data Augmentation

- **Two-level Mix-up**: mix both raw waveforms randomly from the dataset as well as the final spectograms.
- **SpecAugment**: masking up to 48 frequency bins and 192 time frames.
- **Rolling**: roll the waveforms randomly over time.
- **Random Gain**: We multiply the audio waveforms to change the gain by ±7dB.

## 4. Results

- Metric: **mAP** (also referred to as precision-recall area under-curve).
- fine-tune pre-trained PaSST on downstream tasks, removing patch overlap (-N) better than transformer depth (-L).
- Use Patchout: remove half of input sequence.

## 5. Complete Pipeline

1. Input Spectrogram
2. Patch Extraction
3. Linear Projection
4. Disentangled positional encoding
5. Patchout
6. Transformer encoder
7. Classification head (For stem separator, the final classification head will be replaced with a dense reconstruction decoder)

## 6. Reproduce Math Mechanism

Components:
- Spectrogram
- Patch extraction
- Patch embedding
- Positional encoding
- Unstructured Patchout
- Structured Patchout
- Transformer

Dimensions:

- $X \in \mathbb{R}^{1\times 128\times 100}$:
  - 1: one spectrogram channel
  - 128: frequencey bins
  - 100: time frames
- patch size = $16 \times 10$
- patch num = $N = \frac{128}{16} \times \frac{100}{10} = 80$ patches.
- each patch contains $16 \times 10 = 160$ values.
- init patch matrix: $P \in \mathbb{R}^{80 \times 160}$
- embedding dim = 64 so (in `torch`, weight matrix: (out_dim, in_dim)):

$$
E = PW^T + b \\
W \in \mathbb{R}^{64 \times 160}, \quad E \in \mathbb{R}^{80 \times 64}
$$

which means 80 tokens, represented by 64 features, where Patchout enters.

In [1]:
import numpy as np

frequency_bins = 128
time_frames = 100

patch_height = 16
patch_width = 10

spectrogram = np.random.randn(frequency_bins, time_frames)
print("Spectrogram:", spectrogram.shape)

Spectrogram: (128, 100)


### 6.1 Extractor

In [2]:
def extract_patches(spectrogram: np.ndarray, patch_height: int, patch_width: int) -> np.ndarray:
    frequency_bins, time_frames = spectrogram.shape
    if frequency_bins % patch_height != 0:
        raise ValueError("Frequency dimension must be divisible by patch height.")
    if time_frames % patch_width != 0:
        raise ValueError("Time dimension must be divisible by patch width.")

    frequency_patches = frequency_bins // patch_height
    time_patches = time_frames // patch_width

    patches = []
    for i in range(frequency_patches):
        for j in range(time_frames):
            patch = spectrogram[
                i * patch_height : (i+1) * patch_height,
                j * patch_width : (j+1) * patch_width
            ]
            patches.append(patch.flatten())

    return np.stack(patches)

Or remove the loops with `.reshape()` and `.transpose()`.

In [3]:
def extract_patches(
    spectrogram: np.ndarray,
    patch_height: int,
    patch_width: int
) -> np.ndarray:
    frequency_bins, time_frames = spectrogram.shape
    if frequency_bins % patch_height != 0 or time_frames % patch_width != 0:
        raise ValueError("Dimensions must be divisible by patch sizes.")

    freq_patches = frequency_bins // patch_height
    time_patches = time_frames // patch_width

    # Reshape to 4D (freq_patches, patch_height, time_patches, patch_width)
    # Use view to shares the same memory
    patches = spectrogram.reshape(freq_patches, patch_height, time_patches, patch_width)

    # Transpose to group grid dimensions together (freq_patches, time_patches, patch_height, patch_width)
    patches = patches.transpose(0, 2, 1, 3)

    # Flatten
    return patches.reshape(-1, patch_height * patch_width) # (80, 160)

In [4]:
# Veriy
patches = extract_patches(spectrogram, 16, 10)
assert patches.shape == (80, 160)
print("Patches:", patches.shape)

Patches: (80, 160)


The PyTorch

In [5]:
# %load_ext autoreload
# %autoreload 2
# %aimport -torch

In [ ]:
from passt.extractor import PatchExtractor
import torch

extractor = PatchExtractor(
    patch_height=4,
    patch_width=5,
    stride_height=2,
    stride_width=3
)

x = torch.randn(2, 1, 16, 20)
patches, grid = extractor(x)

print(patches.shape)
print(grid)

torch.Size([2, 42, 20])
(7, 6)


### 6.2 Linear Projection

Now,

$$
X_p \in \mathbb{R}^{B \times N \times P}
$$

with:
- $N$: number of patches
- $P = CP_hP_w$: number of values per patch

> But **transformer** does't work with spectrogram patches. -> Embed spectrogram.

$$
E = X_pW + B
$$

with:

$$
W \in \mathbb{R}^{P \times D}
$$

- $D$ := embedding dimension.

> It does not yet understand music. It simply learns a useful coordinate system for representing patches.

### 6.3 Disentangled positional encoding

A **transformer** doesn't know the order or position of input tokens. Say, two identical patches:
- One occurs at 100Hz
- Another occurs at 5 kHz

Their raw patch values could be identical, but their musical meanings are different.

Similarly, two identical patches occurring at different times represent different events.

The paper therefore separates positional information into two learnable embeddings:

$$
E_{f}\in\mathbb{R}^{F_p\times D} \\
E_t \in \mathbb{R}^{T_p \times D}
$$

For a patch at frequency position i and time position j:

$$
Z_{i,j} = E_{i,j} + E_f[i] + E_t[j]
$$

- $E_{i,j}$:= projected patch embedding.

In this Positional embedding, we use `nn.Parameter()`, not `nn.Embedding()`. Why?

Imagine you have a spreadsheet of numbers sitting on your desk.

**`nn.Embedding` is like a librarian.**

If you want row 0, row 1, and row 2 from that spreadsheet, you cannot just grab them. You have to write the numbers `[0, 1, 2]` on a slip of paper, hand that paper to the librarian (`nn.Embedding`), and the librarian goes to the shelf, looks up those row numbers, and hands you back the rows.

**`nn.Parameter` is the spreadsheet itself sitting right on your desk.**

Because it is already right there in front of you as a plain grid of numbers, you don't need a librarian or a slip of paper with row numbers. You just grab the first 3 rows with your hands (`[:3]`) and use them.

In the spectrogram model, the positions are **always sequential starting from zero** (`0, 1, 2, ... up to Fp-1`). They are never shuffled like words in a sentence ("cat" = ID 402, "dog" = ID 19).

Let's compare what your code has to do in forward() using both approaches for a grid of `Fp = 3` frequency patches and `Tp = 4` time patches.

**Approach 1: Using `nn.Embedding` (The Librarian)**

Because `nn.Embedding` is a function that demands integer IDs as input, you have to manufacture fake ID tensors on the fly every single forward pass, make sure they are on the right GPU device, and then manually reshape the outputs so they can be added together:

In [ ]:
import torch.nn as nn

In [ ]:
class WithEmbedding(nn.Module):
    def __init__(self, max_f, max_t, embed_dim):
        super().__init__()
        self.freq_emb = nn.Embedding(max_f, embed_dim)
        self.time_emb = nn.Embedding(max_t, embed_dim)

    def forward(self, x, Fp, Tp):
        # Step 1: Manually create index tensors [0, 1, 2] on the correct device!
        f_idx = torch.arange(Fp, device=x.device) 
        t_idx = torch.arange(Tp, device=x.device)

        # Step 2: Ask the "librarian" to look them up
        f_vec = self.freq_emb(f_idx)  # shape: (Fp, embed_dim)
        t_vec = self.time_emb(t_idx)  # shape: (Tp, embed_dim)

        # Step 3: Manually insert '1' dimensions so they can broadcast
        f_vec = f_vec.unsqueeze(0).unsqueeze(2)  # (1, Fp, 1, embed_dim)
        t_vec = t_vec.unsqueeze(0).unsqueeze(1)  # (1, 1, Tp, embed_dim)

        return f_vec + t_vec


All that extra plumbing: `torch.arange`, `device=x.device`, and four `.unsqueeze()` calls just to get the shapes ready.

**Approach 2: Using `nn.Parameter` (The Raw Tensor)**

With `nn.Parameter`, the tensor is already pre-shaped with those `1`s baked in at initialization. You just slice off what you need and add:

In [ ]:
class WithParameter(nn.Module):
    def __init__(self, max_f, max_t, embed_dim):
        super().__init__()
        # Already shaped (1, max_f, 1, embed_dim)
        self.freq_emb = nn.Parameter(torch.randn(1, max_f, 1, embed_dim) * 0.02)
        self.time_emb = nn.Parameter(torch.randn(1, 1, max_t, embed_dim) * 0.02)

    def forward(self, x, Fp, Tp):
        # No index tensors, no device matching, no unsqueezing!
        return self.freq_emb[:, :Fp, :, :] + self.time_emb[:, :, :Tp, :]

**Conclusion**

- Use `nn.Embedding` when you have arbitrary, **out-of-order IDs** (like word tokens `[54, 912, 7]`) that need a lookup table.
- Use `nn.Parameter` when you want a **continuous block of weights** that you will slice, multiply, or add directly in math operations without doing index lookups

### 6.4 Patchout

### 6.4.1 Unstructured Patchout

Randomly retain X% of the tokens.

### 6.4.2 Structured Patchout

### 6.5 Transformer encoder

- input shape: (B, K, D) (K:= number or retained patches)
- successive self-attention and feed-forward blocsk
- A transformer block consists of:
  1. Layer normalisation
  2. Multi-head self-attention
  3. Residual connection
  4. Layer normalisation
  5. Feed-forward network
  6. Residual connection

- Core calculation:

$$
\text{Attention}(Q,K,V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V
$$

For B=2, K=80, D=128. If we use 4 heads:

$$
d_k = \frac{128}{4} = 32
$$

Attention matrix for each head is 80x80 (The original sequence would have an attention matrix of 160x160). So it is 6400 rather than 25,600 elements per head.

> This is a conventional Transformer encoder implementation inspired by the PaSST architecture, not a bit-for-bit reproduction of its DeiT-based implementation.

## 7. The Part that the paper doesn't solve

PaSST was designed for audio classification.

Its classification token aggregates information from the spectrogram, and the classifier predicts audio labels.

We need something different.

Suppose our input is a spectrogram:

$$
X \in \mathbb{R}^{F\times T}
$$

We want vocals, drums, bass, other:

$$
\^{X}_1, \^{X}_2, \^{X}_3, \^{X}_4
$$

We cannot simply send the reduced Transformer output into a classification head. We need to reconstruct the **full spatial arrangement of the patches**.

### 7.1 Scatter retained tokens back into their original positions

Suppose we originally have 160 patches, but Patchout retains 80.

The Transformer returns:
$$
H\in\mathbb{R}^{B\times80\times D}
$$

We create a full grid containing 160 positions.

The 80 encoded tokens are placed back into their original positions. The missing positions receive a learnable mask embedding.

### 7.2 Why a decoder is necessary

After restoring the grid:
$$
H_{\text{full}}\in\mathbb{R}^{B\times N\times D}
$$

We reshape into:

$$
(B, D, F_p, T_p)
$$

Then use a convolutional decoder to produce dense predictions.

**Why convolution?**

Because neighbouring patches have spatial relationships. A convolutional decoder can use neighbouring encoded features to help reconstruct regions whose tokens were removed.

**The Design**

1. Restored token grid (B, D, Fp, Tp)
2. Conv2D + GELU
3. Conv2D + GELU
4. Dense output projection
5. Vocals + Drums + Bass + Other

This is a proposed adaptation, not a component specified by the Patchout paper.

One important trade-off: a full Transformer decoder would reintroduce quadratic attention over all tokens. A lightweight convolutional decoder avoids that particular cost.

For overlapping patches, we also need to reconstruct the spectrogram using overlap-add or an equivalent folding operation, with normalisation for regions covered by multiple patches.

For the first implementation, I would use non-overlapping patches in the reconstruction stage. We can introduce overlap once the baseline works.

## 8. Training objective: what should the model learn?

The Patchout paper trains an audio classifier using classification objectives. It does not specify a source-separation loss.

We need paired mixtures and ground-truth stems.

Given:

$$
x(t) = \sum_{s=1}^S y_s(t)
$$

Model predicts:

$$
\^{y}_1(t),...,\^{y}_S(t)
$$

A basic waveform reconstruction objective is:

$$
\mathcal{L}_{\text{wave}} = \sum_{s=1}^S\| \^{y}_s - y_s \|_1
$$

We could also investigate (*but one at a time*):
- SI-SDR loss for source reconstruction quality.
- Spectral reconstruction loss.
- Multi-resolution STFT loss.
- Mixture consistency, ensuring predicted stems sum back to the mixture.

> **A crucial design decision**: Are we predicting complex STFT masks, magnitude masks with mixture phase, or waveforms directly? That choice affects the decoder and reconstruction pipeline.

## 9. Putting it all together

The encoder below is not a complete separator. It assumes a single input channel and uses non-overlapping-compatible patch dimensions, although the extractor supports overlap.

We will attach the restoration and dense decoder components next.

A few additional details from the paper worth retaining:
- The experiments use 32 kHz mono audio, 128 Mel bands, a 25 ms window and 10 ms hop.
- The authors use AdamW and a maximum learning rate of $10^{-5}$ in their reported training setup.
- They report roughly half the input sequence being removed during fine-tuning.
- Structured Patchout generally provides substantial computational savings, although the benefit varies by task.

Input:

$$
X \in \mathbb{R}^{2\times 1\times 64\times 100}
$$

Parameters:
- Patch size: $8\times10$.
- Stride: $4\times5$.
- Embedding dimension: 128.
- Patchout ratio: 0.5.
- Embedding dimension: 128.
- Attention heads: 4.
- Transformer layers: 2–4.

**Questions**

1. What are the frequency and time patch counts?
2. What is the complete output shape of `PatchExtractor`?
3. What is the shape after linear projection?
4. What is the shape after positional encoding?
5. What is the shape after unstructured Patchout?
6. What is the size of the attention matrix per head before and after Patchout?
7. Why must we preserve the original indices when removing patches?

**One final research question**: If we randomly remove half the patches, how can the decoder reconstruct information that it never received? Is the decoder genuinely reconstructing the missing information, or merely estimating it from neighbouring context and learned priors?

**Answers**

1. Freq patches = 15, time patches = 19
2. Complete output shape of PatchExtractor: (B, Fp*Tp, C*ph*pw) = (2, 285, 80)
3. Linear projection output: (B, N, D) = (2, 285, 128)
4. Positional encoding output: (B, N, D) = (2, 285, 128)
5. Unstructured patchout output: (B, N, D) = (2, 285, 128)
6. Attention matrix per head before patchout: NxN = 285x285, after patchout: KxK = 142x142.
7. We need to preserve the original indices when removing patches because they will be used in decoder/reconstruction for my project.

Final research question: The model will estimate the missed patches, but I think like in U-Net, there'll be skip connection to make it more precise.

In [5]:
import torch
from passt.pae import PatchoutAudioEncoder

In [9]:
B = 2     # Batch size (4 audio clips)
C = 1     # Channels (1 for mono audio spectrogram)
F = 64   # Frequency bins (e.g., 128 Mel bins)
T = 100   # Time frames

spectrogram = torch.randn(B, C, F, T)

print("Spectrogram:", spectrogram.shape)

Spectrogram: torch.Size([2, 1, 64, 100])


In [10]:
model = PatchoutAudioEncoder(
    patch_height=8,
    patch_width=10,
    stride_height=4,
    stride_width=5,
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    dropout_ratio=.5
)

encoded, indices, grid = model.forward(spectrogram)

print("Encoded:", encoded.shape)
print("Indices:", indices.shape)
print("grid:")
print(grid)

Encoded: torch.Size([2, 142, 128])
Indices: torch.Size([2, 142])
grid:
(15, 19)


D:\work\code\in-progress\music-stem-separator\src\passt\transformer.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


In [ ]:
# Set model training flag to False -> evaluation mode
model.eval()

PatchoutAudioEncoder(
  (extractor): PatchExtractor()
  (embedding): PatchEmbedding(
    (projection): Linear(in_features=80, out_features=128, bias=True)
  )
  (positional_encoding): FactorisedPositionalEncoding()
  (patchout): UnstructuredPatchout()
  (transformer): TransformerEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropou